In [1]:
import os
from pathlib import Path

# Set the cache before importing TensorFlow. It stays outside the submission folder.
os.environ.setdefault("KERAS_HOME", str(Path.home() / ".keras"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import csv
import gc
import hashlib
import json
import platform
import time
from importlib.metadata import version

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import tensorflow as tf
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

print("TensorFlow:", tf.__version__)
print("Visible devices:", tf.config.list_physical_devices())


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
SEED = 42
NUM_CLASSES = 100
IMAGE_SIZE = 32
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-4
VALIDATION_FRACTION = 0.10
COMMON_ACCURACY = 0.20
PATCH_SIZE = 8
EMBED_DIM = 192
NUM_HEADS = 6
NUM_BLOCKS = 6
MLP_DIM = 768
DROPOUT = 0.10
RESULTS_DIR = Path.cwd() / "assignment4_results_32x32-1"

assert IMAGE_SIZE >= 32 and IMAGE_SIZE % PATCH_SIZE == 0
assert EMBED_DIM % NUM_HEADS == 0
assert EPOCHS > 0 and BATCH_SIZE > 0

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass  # Already initialized by this kernel.
DEVICE = "/GPU:0" if gpus else "/CPU:0"
tf.keras.mixed_precision.set_global_policy("float32")
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
if not gpus:
    print("Training device: CPU.")
print("All experiments use", DEVICE)


In [ ]:
from tensorflow.keras.datasets import cifar100 
from tensorflow.keras.utils import to_categorical 
 
# Load the CIFAR-100 dataset 
(x_train, y_train), (x_test, y_test) = cifar100.load_data() 
 
# Normalize the pixel values (0 to 255) to (0 to 1) 
x_train = x_train.astype('float32') / 255.0 
x_test = x_test.astype('float32') / 255.0 
 
# Convert class labels to one-hot encoded vectors 
y_train = to_categorical(y_train, 100) 
y_test = to_categorical(y_test, 100) 
 
print(f"x_train shape: {x_train.shape}") 
print(f"y_train shape: {y_train.shape}") 
print(f"x_test shape: {x_test.shape}") 
print(f"y_test shape: {y_test.shape}") 


In [ ]:
def prepare_batch(images, labels):
    images = tf.cast(images, tf.float32) * 255.0
    return tf.keras.applications.vgg16.preprocess_input(images), tf.cast(labels, tf.float32)

def make_dataset(images, labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        dataset = dataset.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.map(prepare_batch, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    options = tf.data.Options()
    options.experimental_deterministic = True
    return dataset.with_options(options).prefetch(tf.data.AUTOTUNE)

assert x_train.shape == (50000, 32, 32, 3) and x_test.shape == (10000, 32, 32, 3)
train_indices, val_indices = train_test_split(
    np.arange(len(x_train)), test_size=VALIDATION_FRACTION,
    stratify=y_train.argmax(axis=1), random_state=SEED)
assert not np.intersect1d(train_indices, val_indices).size
assert len(train_indices) == 45000 and len(val_indices) == 5000
assert np.all(np.bincount(y_train[train_indices].argmax(1), minlength=100) == 450)
assert np.all(np.bincount(y_train[val_indices].argmax(1), minlength=100) == 50)
SPLIT_HASH = hashlib.sha256(
    train_indices.astype("<i8").tobytes() + val_indices.astype("<i8").tobytes()).hexdigest()
print("Train / validation / test:", len(train_indices), len(val_indices), len(x_test))


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import shutil

MODEL_NAMES = ("vgg16_transfer", "vgg16_scratch", "vit_scratch")
LABELS = { "vgg16_transfer": "VGG16 transfer learning", "vgg16_scratch": "VGG16 from scratch",
          "vit_scratch": "ViT from scratch"}
HISTORY_FIELDS = ("model", "epoch", "loss", "accuracy", "val_loss", "val_accuracy",
                  "test_loss", "test_accuracy", "epoch_seconds", "test_seconds", "cumulative_seconds")
SUMMARY_FIELDS = ("model", "parameters", "trainable_parameters", "final_train_accuracy",
                  "final_val_accuracy", "test_loss", "test_accuracy", "total_epoch_seconds",
                  "total_test_seconds", "mean_epoch_seconds", "mean_epoch_seconds_after_first",
                  "target_accuracy", "target_epoch", "target_seconds")

def synchronize():
    tf.test.experimental.sync_devices()

def write_csv(path, rows, fields):
    with Path(path).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

class EpochRecorder(tf.keras.callbacks.Callback):
    def __init__(self, name, test_ds, path):
        super().__init__()
        self.name, self.test_ds, self.path = name, test_ds, Path(path)
        self.rows, self.cumulative = [], 0.0

    def on_epoch_begin(self, epoch, logs=None):
        synchronize()
        self.started = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        synchronize()
        seconds = time.perf_counter() - self.started
        self.cumulative += seconds
        row = {"model": self.name, "epoch": epoch+1,
               **{key: float(logs[key]) for key in ("loss", "accuracy", "val_loss", "val_accuracy")},
               "epoch_seconds": seconds, "cumulative_seconds": self.cumulative}
        # Test evaluation happens after stopping the training/validation timer.
        test_started = time.perf_counter()
        test = self.model.evaluate(self.test_ds, verbose=0, return_dict=True)
        synchronize()
        row.update(test_loss=float(test["loss"]), test_accuracy=float(test["accuracy"]),
                   test_seconds=time.perf_counter()-test_started)
        if not np.all(np.isfinite([row[key] for key in HISTORY_FIELDS[2:]])):
            raise FloatingPointError(f"Non-finite measurements in {self.name}, epoch {epoch+1}")
        self.rows.append(row)
        exists = self.path.exists()
        with self.path.open("a", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=HISTORY_FIELDS)
            if not exists:
                writer.writeheader()
            writer.writerow(row)
        logs.update(test_loss=row["test_loss"], test_accuracy=row["test_accuracy"])

def train_one(name, builder, train_ds, val_ds, test_ds, history_path=None):
    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    with tf.device(DEVICE):
        model = builder()
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                      loss="categorical_crossentropy", metrics=["accuracy"], jit_compile=False)
        model.summary()
        recorder = EpochRecorder(name, test_ds,
                                 Path(history_path) if history_path is not None else RESULTS_DIR/"history.csv")
        model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[recorder], verbose=2)
        assert len(recorder.rows) == EPOCHS
        times = [row["epoch_seconds"] for row in recorder.rows]
        last = recorder.rows[-1]
        reached = next((row for row in recorder.rows if row["val_accuracy"] >= COMMON_ACCURACY), None)
        summary = {"model": name, "parameters": model.count_params(),
                   "trainable_parameters": int(sum(np.prod(w.shape) for w in model.trainable_weights)),
                   "final_train_accuracy": last["accuracy"], "final_val_accuracy": last["val_accuracy"],
                   "test_loss": last["test_loss"], "test_accuracy": last["test_accuracy"],
                   "total_epoch_seconds": float(sum(times)),
                   "total_test_seconds": float(sum(row["test_seconds"] for row in recorder.rows)),
                   "mean_epoch_seconds": float(np.mean(times)),
                   "mean_epoch_seconds_after_first": float(np.mean(times[1:])) if len(times)>1 else None,
                   "target_accuracy": COMMON_ACCURACY,
                   "target_epoch": reached["epoch"] if reached else None,
                   "target_seconds": reached["cumulative_seconds"] if reached else None}
    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return recorder.rows, summary


def run_model(name, builder, rerun=False):
    if name not in MODEL_NAMES:
        raise ValueError(f"Unknown model: {name}")
    replacing = name in summaries
    if replacing and not rerun:
        raise ValueError(f"{name} is already recorded. Use rerun=True to repeat it.")
    if not replacing:
        expected = next((model for model in MODEL_NAMES if model not in summaries), None)
        if name != expected:
            raise ValueError("Model order does not match the experiment record.")

    # All retained models must belong to the same experimental settings.
    for key, value in (("seed", SEED), ("image_size", IMAGE_SIZE),
                       ("batch_size", BATCH_SIZE), ("epochs", EPOCHS),
                       ("learning_rate", LEARNING_RATE), ("device", DEVICE)):
        if manifest.get(key) != value:
            raise ValueError(f"The current {key} differs from this run's settings.")

    history_file = RESULTS_DIR/"history.csv"
    history_before = history_file.read_bytes() if history_file.exists() else None
    previous_rows = []
    if history_file.exists():
        with history_file.open(newline="", encoding="utf-8") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != list(HISTORY_FIELDS):
                raise ValueError("The saved history has a different CSV format.")
            previous_rows = list(reader)

    token = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
    archive = None
    if replacing:
        archive = RESULTS_DIR/"previous_runs"/(name+"_"+token)
        archive.mkdir(parents=True, exist_ok=False)
        for saved in RESULTS_DIR.iterdir():
            if saved.is_file() and saved.suffix in (".csv", ".json", ".md", ".png") and not saved.name.startswith("."):
                shutil.copy2(saved, archive/saved.name)
        print("Previous results archived in", archive)

    staged_history = RESULTS_DIR/(".history_"+name+"_"+token+".csv")
    tf.keras.utils.set_random_seed(SEED)
    train_ds = make_dataset(x_train[train_indices], y_train[train_indices], training=True)
    val_ds = make_dataset(x_train[val_indices], y_train[val_indices])
    test_ds = make_dataset(x_test, y_test)
    try:
        rows, summary = train_one(name, builder, train_ds, val_ds, test_ds,
                                  history_path=staged_history)
        if len(rows) != EPOCHS or [row["epoch"] for row in rows] != list(range(1,EPOCHS+1)):
            raise ValueError("The new model history is incomplete.")
        if any(row["model"] != name for row in rows) or summary["model"] != name:
            raise ValueError("The new model measurements have an incorrect label.")
        current = history_file.read_bytes() if history_file.exists() else None
        if current != history_before:
            raise RuntimeError("The saved history changed during training; new measurements remain in the temporary CSV.")

        retained = [row for row in previous_rows if row["model"] != name]
        combined = retained+rows
        combined.sort(key=lambda row: (MODEL_NAMES.index(row["model"]), int(row["epoch"])))
        updated_summaries = {**summaries, name:summary}
        ordered_summaries = [updated_summaries[model] for model in MODEL_NAMES if model in updated_summaries]
        updated_manifest = dict(manifest)
        updated_manifest["completed_models"] = [model for model in MODEL_NAMES if model in updated_summaries]
        updated_manifest["status"] = "complete" if len(updated_summaries)==len(MODEL_NAMES) else "running"
        if archive is not None:
            updated_manifest["latest_replacements"] = dict(manifest.get("latest_replacements", {}))
            updated_manifest["latest_replacements"][name] = str(archive.relative_to(RESULTS_DIR))

        staged_summary = RESULTS_DIR/(".summary_"+token+".csv")
        staged_manifest = RESULTS_DIR/(".run_"+token+".json")
        write_csv(staged_history, combined, HISTORY_FIELDS)
        write_csv(staged_summary, ordered_summaries, SUMMARY_FIELDS)
        staged_manifest.write_text(json.dumps(updated_manifest, indent=2, default=str), encoding="utf-8")
        staged_history.replace(history_file)
        staged_summary.replace(RESULTS_DIR/"summary.csv")
        staged_manifest.replace(RESULTS_DIR/"run.json")
        histories[name] = rows
        summaries[name] = summary
        manifest.clear()
        manifest.update(updated_manifest)

        # These derived files must be regenerated from the updated measurements.
        if replacing:
            stale = [f"figure_{MODEL_NAMES.index(name)+1}_{name}.png", "report.md",
                     "figure_4_test_accuracy_comparison.png", "figure_5_test_loss_comparison.png",
                     "figure_6_learning_speed.png", "figure_7_epoch_time.png"]
            for filename in stale:
                (RESULTS_DIR/filename).unlink(missing_ok=True)
        print("Measurements saved for", name)
    finally:
        del train_ds, val_ds, test_ds
        gc.collect()


In [ ]:
def finish_figure(fig, filename):
    fig.savefig(RESULTS_DIR/filename, dpi=170)
    plt.show()
    plt.close(fig)

def style_axis(ax, ylabel):
    ax.set(xlabel="Epoch", ylabel=ylabel)
    if "accuracy" in ylabel.lower():
        ax.set_ylim(0, 100)
    else:
        ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.25)
    ax.legend()

def plot_model(name):
    rows = histories[name]
    epochs = [row["epoch"] for row in rows]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    for ax, metric in zip(axes, ("accuracy", "loss")):
        scale = 100 if metric=="accuracy" else 1
        for prefix, label, color, linestyle in (
                ("", "Training", "tab:blue", "-"),
                ("val_", "Validation", "tab:orange", "--"),
                ("test_", "Test", "tab:green", "-.")):
            ax.plot(epochs, [scale*row[prefix+metric] for row in rows],
                    label=label, color=color, linestyle=linestyle)
        ylabel = "Accuracy (%)" if metric=="accuracy" else "Cross-entropy loss"
        style_axis(ax, ylabel)
        ax.set_title("Accuracy per epoch" if metric=="accuracy" else "Loss per epoch")
    number = MODEL_NAMES.index(name)+1
    fig.suptitle(f"Figure {number}. {LABELS[name]}")
    finish_figure(fig, f"figure_{number}_{name}.png")

def model_results_text(name):
    row = summaries[name]
    lines = [f"### Results for {LABELS[name]}", "",
             f"I obtained final validation accuracy of {100*row['final_val_accuracy']:.2f}% "
             f"and final test accuracy of {100*row['test_accuracy']:.2f}%. "
             f"The final test loss was {row['test_loss']:.4f}.", "",
             "| Epoch | Training accuracy | Validation accuracy | Test accuracy | Training loss | Validation loss | Test loss |",
             "|---:|---:|---:|---:|---:|---:|---:|"]
    for epoch in sorted({1, max(1,EPOCHS//2), EPOCHS}):
        r = histories[name][epoch-1]
        lines.append(f"| {epoch} | {100*r['accuracy']:.2f}% | {100*r['val_accuracy']:.2f}% | "
                     f"{100*r['test_accuracy']:.2f}% | {r['loss']:.4f} | {r['val_loss']:.4f} | {r['test_loss']:.4f} |")
    gap = 100*(row['final_train_accuracy']-row['final_val_accuracy'])
    lines += ["", f"The final training minus validation accuracy gap was {gap:+.2f} percentage points. "
              "I consider this gap together with the loss curves when assessing overfitting."]
    return "\n".join(lines)

def plot_test_comparisons():
    for number, metric in ((4,"accuracy"),(5,"loss")):
        fig, ax = plt.subplots(figsize=(8,4), constrained_layout=True)
        for name in MODEL_NAMES:
            rows = histories[name]
            scale = 100 if metric=="accuracy" else 1
            ax.plot([r["epoch"] for r in rows], [scale*r["test_"+metric] for r in rows], label=LABELS[name])
        style_axis(ax, "Test accuracy (%)" if metric=="accuracy" else "Test cross-entropy loss")
        ax.set_title(f"Figure {number}. Test {metric} per epoch for all models")
        finish_figure(fig, f"figure_{number}_test_{metric}_comparison.png")

def plot_learning_speed():
    fig, axes = plt.subplots(1,2,figsize=(12,4), constrained_layout=True)
    for name in MODEL_NAMES:
        rows = histories[name]
        values = [100*r["val_accuracy"] for r in rows]
        axes[0].plot([r["epoch"] for r in rows],values,label=LABELS[name])
        axes[1].plot([r["cumulative_seconds"]/60 for r in rows],values,label=LABELS[name])
    for ax, xlabel in zip(axes,("Epoch","Cumulative training + validation time (minutes)")):
        ax.axhline(100*COMMON_ACCURACY,color="gray",linestyle=":",label="Common target")
        ax.set(xlabel=xlabel,ylabel="Validation accuracy (%)")
        ax.set_ylim(0, 100)
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.suptitle("Figure 6. Validation accuracy and learning speed")
    finish_figure(fig,"figure_6_learning_speed.png")
    fig, ax = plt.subplots(figsize=(8,4),constrained_layout=True)
    for name in ("vgg16_scratch","vit_scratch"):
        rows=histories[name]
        ax.plot([r["epoch"] for r in rows],[r["epoch_seconds"] for r in rows],marker=".",label=LABELS[name])
    style_axis(ax,"Training + validation time (seconds)")
    ax.set_title("Figure 7. Epoch time; epoch 1 includes tracing and warm-up")
    finish_figure(fig,"figure_7_epoch_time.png")

def comparison_text():
    scratch, transfer, vit = [summaries[name] for name in MODEL_NAMES]
    gain = 100*(transfer['test_accuracy']-scratch['test_accuracy'])
    val_gain = 100*(transfer['final_val_accuracy']-scratch['final_val_accuracy'])
    difference = 100*(vit['final_val_accuracy']-scratch['final_val_accuracy'])
    winner = "ViT" if difference>0 else "VGG16 from scratch" if difference<0 else "Neither model (equal accuracy)"
    lines = ["## Comparison of results", "",
             "| Model | Parameters | Final validation accuracy | Final test accuracy | Final test loss | Mean epoch seconds |",
             "|---|---:|---:|---:|---:|---:|"]
    for name in MODEL_NAMES:
        r=summaries[name]
        lines.append(f"| {LABELS[name]} | {r['parameters']:,} | {100*r['final_val_accuracy']:.2f}% | "
                     f"{100*r['test_accuracy']:.2f}% | {r['test_loss']:.4f} | {r['mean_epoch_seconds']:.2f} |")
    outcome = "improved" if gain>0 else "reduced" if gain<0 else "did not change"
    lines += ["", "### Exercise 2 Effect of transfer learning", "",
              f"Transfer learning {outcome} final test accuracy by {abs(gain):.2f} percentage points. "
              f"Its final validation accuracy difference was {val_gain:+.2f} percentage points. "
              "As discussed in Lecture 5, pretrained visual features can reduce the learning required "
              "for a new task. This measured result describes their benefit or limitation under my training settings.",
              "", "### Exercise 3 Comparison with VGG16 from scratch", "",
              f"Model with the higher final validation accuracy: {winner}. "
              f"ViT minus VGG16 final validation accuracy was {difference:+.2f} percentage points. "
              "I compare the final epoch rather than the maximum on either curve. "
              "Convolution supplies local shared filters, while attention learns relationships between patches "
              "(Lectures 3, 7 and 9). The different parameter counts also affect this comparison.",
              "", "### Learning speed", "",
              f"I compare time and epochs to the preselected target of {100*COMMON_ACCURACY:.1f}% validation accuracy.", ""]
    for name in MODEL_NAMES:
        r=summaries[name]
        reached=(f"epoch {int(r['target_epoch'])}, {r['target_seconds']/60:.2f} minutes"
                 if r['target_epoch'] is not None else "not reached within the training budget")
        lines.append(f"- {LABELS[name]}: {reached}.")
    if scratch['target_seconds'] is not None and vit['target_seconds'] is not None:
        a,b=scratch['target_seconds'],vit['target_seconds']
        faster="ViT" if b<a else "VGG16" if a<b else "Neither model (equal measured time)"
        lines += ["",f"Model reaching the target in less measured time: {faster}."]
    else:
        lines += ["","At least one scratch model did not reach the target, so a time-to-target comparison for both is unavailable."]
    lines += ["",f"Mean epoch times were {scratch['mean_epoch_seconds']:.2f} seconds for VGG16 "
              f"and {vit['mean_epoch_seconds']:.2f} seconds for ViT."]
    if EPOCHS>1:
        lines.append(f"Excluding epoch 1, the means were {scratch['mean_epoch_seconds_after_first']:.2f} "
                     f"and {vit['mean_epoch_seconds_after_first']:.2f} seconds, respectively.")
    lines += ["","Test evaluation time is recorded separately and excluded from these epoch times. "
              "Faster epochs alone do not establish faster learning. My results use one seed and shared "
              "hyperparameters, which may not be optimal for each architecture."]
    return "\n".join(lines)

def save_report():
    assert manifest["status"]=="complete" and set(summaries)==set(MODEL_NAMES)
    lines = ["# Assignment 4 Charis Aletraris", "", "## Experiment setup", "",
             f"I used the original {IMAGE_SIZE} x {IMAGE_SIZE} CIFAR-100 images with no resizing, "
             f"a {len(train_indices):,}/{len(val_indices):,}/{len(x_test):,} training/validation/test split, "
             f"seed {SEED}, batch size {BATCH_SIZE}, {EPOCHS} epochs and Adam learning rate {LEARNING_RATE}. "
             f"All models used the same VGG16 channel preprocessing and device {DEVICE}. "
             "I recorded test metrics after each epoch without using them for weight updates or model selection.", "",
             "## Architectures", "",
             "I used the VGG16 convolutional backbone with global average pooling and a 100-class softmax. "
             "Its 32 x 32 inputs produce a 1 x 1 x 512 final feature map. The scratch model used random "
             "weights; the transfer model used ImageNet weights. All layers were trainable in both models.", "",
             f"My ViT used {PATCH_SIZE} x {PATCH_SIZE} patches ({(IMAGE_SIZE//PATCH_SIZE)**2} tokens), "
             f"{EMBED_DIM} features, {NUM_BLOCKS} pre-normalized encoder blocks, {NUM_HEADS} attention heads "
             f"with {EMBED_DIM//NUM_HEADS} features per head, GELU feed-forward layers of sizes "
             f"{MLP_DIM} and {EMBED_DIM}, dropout {DROPOUT}, residual additions, learned positional embeddings, "
             "final LayerNorm, mean pooling and a 100-class softmax. I normalized patch projections before "
             "adding positions. I selected this compact architecture to limit attention cost and training time. "
             "It used no pretrained weights."]
    for i,name in enumerate(MODEL_NAMES,1):
        lines += ["",f"## Exercise {i} {LABELS[name]}","",model_results_text(name),"",
                  f"![Figure {i}](figure_{i}_{name}.png)","",
                  f"Figure {i}. Left: accuracy; right: loss. Training: solid blue; validation: dashed orange; test: dash-dot green."]
    lines += ["",comparison_text()]
    for i,filename,caption in (
            (4,"test_accuracy_comparison","Test accuracy per epoch for all three models."),
            (5,"test_loss_comparison","Test loss per epoch for all three models."),
            (6,"learning_speed","Validation accuracy against epochs and cumulative training and validation time."),
            (7,"epoch_time","Training and validation time per epoch for VGG16 and ViT from scratch.")):
        lines += ["",f"![Figure {i}](figure_{i}_{filename}.png)","",f"Figure {i}. {caption}"]
    lines += ["","Sources of measurements: history.csv, summary.csv and run.json. "
              "Conceptual sources: course Lectures 3, 4, 5, 7 and 9."]
    (RESULTS_DIR/"report.md").write_text("\n".join(lines)+"\n",encoding="utf-8")


In [ ]:
if RESULTS_DIR.exists() and any(RESULTS_DIR.iterdir()):
    raise FileExistsError(f"{RESULTS_DIR} already contains results. Choose a new RESULTS_DIR.")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
manifest = {
    "status": "running", "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE, "learning_rate": LEARNING_RATE, "optimizer": "Adam",
    "loss": "categorical_crossentropy", "augmentation": "none", "precision": "float32",
    "preprocessing": "original 32x32 float RGB [0,1] -> multiply by255 -> VGG16 preprocess_input; no resize",
    "split": {"train": len(train_indices), "validation": len(val_indices), "test": len(x_test),
              "validation_fraction": VALIDATION_FRACTION, "indices_sha256": SPLIT_HASH},
    "vit": {"patch_size": PATCH_SIZE, "patches": (IMAGE_SIZE//PATCH_SIZE)**2,
            "embed_dim": EMBED_DIM, "heads": NUM_HEADS, "blocks": NUM_BLOCKS,
            "mlp_dim": MLP_DIM, "dropout": DROPOUT, "pooling": "mean", "pretrained": False,
            "patch_projection_normalization": "LayerNorm before positional embeddings"},
    "vgg": {"backbone": "VGG16 include_top=False", "head": "GAP -> Dense100 softmax",
            "all_layers_trainable": True, "transfer_weights": "imagenet"},
    "common_accuracy_target": COMMON_ACCURACY, "device": DEVICE,
    "device_details": [tf.config.experimental.get_device_details(g) for g in gpus],
    "platform": platform.platform(), "processor": platform.processor(),
    "python": platform.python_version(),
    "versions": {p: version(p) for p in ("tensorflow", "keras", "numpy", "scikit-learn", "matplotlib")},
    "deterministic_ops": True, "timing": "synchronized training + validation; test evaluation timed separately; epoch1 includes tracing",
    "test_policy": "evaluate after each epoch for curves; no test-driven optimization, stopping or selection",
    "model_order": list(MODEL_NAMES)}
(RESULTS_DIR / "run.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")

histories, summaries = {}, {}
manifest["completed_models"] = []


In [ ]:
def build_vgg(pretrained=False):
    backbone = tf.keras.applications.VGG16(
        weights="imagenet" if pretrained else None,
        include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    backbone.trainable = True
    inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
    features = backbone(inputs)
    pooled = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pool")(features)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax",
                                    name="cifar100_classifier")(pooled)
    return tf.keras.Model(inputs, outputs,
                          name="vgg16_transfer" if pretrained else "vgg16_scratch")


In [ ]:
run_model("vgg16_scratch", lambda: build_vgg(pretrained=False))


In [ ]:
plot_model("vgg16_scratch")
display(Markdown(model_results_text("vgg16_scratch")))


In [ ]:
run_model("vgg16_transfer", lambda: build_vgg(pretrained=True))

In [ ]:
plot_model("vgg16_transfer")
display(Markdown(model_results_text("vgg16_transfer")))


In [ ]:
class PatchPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, image_size, patch_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.image_size, self.patch_size, self.embed_dim = image_size, patch_size, embed_dim
        self.num_patches = (image_size // patch_size) ** 2
        self.projection = tf.keras.layers.Conv2D(
            embed_dim, kernel_size=patch_size, strides=patch_size, padding="valid")
        self.token_norm = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.position = tf.keras.layers.Embedding(self.num_patches, embed_dim)

    def call(self, images):
        projected = self.projection(images)
        tokens = tf.reshape(projected, (-1, self.num_patches, self.embed_dim))
        return self.token_norm(tokens) + self.position(tf.range(self.num_patches))

    def get_config(self):
        return {**super().get_config(), "image_size": self.image_size,
                "patch_size": self.patch_size, "embed_dim": self.embed_dim}

def build_vit():
    inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
    tokens = PatchPositionEmbedding(IMAGE_SIZE, PATCH_SIZE, EMBED_DIM,
                                     name="patch_and_position")(inputs)
    tokens = tf.keras.layers.Dropout(DROPOUT)(tokens)
    for block in range(NUM_BLOCKS):
        normalized = tf.keras.layers.LayerNormalization(epsilon=1e-6)(tokens)
        attended = tf.keras.layers.MultiHeadAttention(
            num_heads=NUM_HEADS, key_dim=EMBED_DIM // NUM_HEADS,
            dropout=DROPOUT, name=f"attention_{block+1}")(normalized, normalized)
        tokens = tf.keras.layers.Add()([tokens, attended])
        normalized = tf.keras.layers.LayerNormalization(epsilon=1e-6)(tokens)
        transformed = tf.keras.layers.Dense(MLP_DIM, activation="gelu")(normalized)
        transformed = tf.keras.layers.Dropout(DROPOUT)(transformed)
        transformed = tf.keras.layers.Dense(EMBED_DIM)(transformed)
        transformed = tf.keras.layers.Dropout(DROPOUT)(transformed)
        tokens = tf.keras.layers.Add()([tokens, transformed])
    tokens = tf.keras.layers.LayerNormalization(epsilon=1e-6)(tokens)
    pooled = tf.keras.layers.GlobalAveragePooling1D()(tokens)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(pooled)
    return tf.keras.Model(inputs, outputs, name="vit_scratch")


In [ ]:
run_model("vit_scratch", build_vit)


In [ ]:
plot_model("vit_scratch")
display(Markdown(model_results_text("vit_scratch")))


In [ ]:
plot_test_comparisons()


In [ ]:
plot_learning_speed()
display(Markdown(comparison_text()))
save_report()
